### TODO:
COMENZAR YA CON EL MODELO DE DIFUSUION PARA AUDIO, PERO ANTES ARREGLAR LO DE LAS CAPAS DEL MODELO

In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

import torchaudio.transforms as T
import math   
from src.dataset import NSynth
import torch
from src.diffusion import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = r"C:\Users\Articuno\Desktop\TFG-info\data\models\mod_diff.pth"
sch_path =  r"C:\Users\Articuno\Desktop\TFG-info\data\models\sch_diff.pth"
temp_path = r"C:\Users\Articuno\Desktop\TFG-info\data\models\temp_diff.pth"
sch_temp_path = r"C:\Users\Articuno\Desktop\TFG-info\data\models\temp_sch_diff.pth" 

In [15]:
@torch.no_grad()
def sample_images(model, scheduler, num_images=4, image_size=(1, 28, 28)):
    model.eval()

    x = torch.randn(num_images, *image_size, device=device)
    T = scheduler.beta.size(0)

    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar

    for t in reversed(range(T)):
        t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)

        # predicción de ruido
        e_pred = model(x, t_batch)
        
        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        
        # beta_t = beta_t.view(1,1,1,1)
        # alpha_t = alpha_t.view(1,1,1,1)
        # alpha_bar_t = alpha_bar_t.view(1,1,1,1)


        # Coeficientes DDPM
        coef1 = 1 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

        mu = coef1 * (x - coef2 * e_pred)

        if t > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mu + sigma_t * noise
        else:
            x = mu

    x = x.clamp(0, 1).cpu()

    # Mostrar imágenes
    fig, axes = plt.subplots(1, num_images, figsize=(num_images*2, 2))
    for i in range(num_images):
        axes[i].imshow(x[i, 0], cmap='gray')
        axes[i].axis('off')
    plt.show()

    return x

@torch.no_grad()
def remove_noise(model, scheduler, image_size=(1, 28, 28), x=None, alpha=0):
    model.eval()
    if x is None:
        x = torch.randn(1, *image_size, device=device)
        alpha = 1

    # Guardar imagen original
    x_original = x.clone().cpu()

    # Añadir ruido
    noise = torch.randn(1, *image_size, device=device) * (1 - alpha)
    x_noisy = x * alpha + noise
    x = x_noisy.clone()  # este será el tensor que pasaremos al DDPM

    T = scheduler.beta.size(0)
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar

    for t in reversed(range(T)):
        t_batch = torch.full((1,), t, device=device, dtype=torch.long)
        e_pred = model(x, t_batch)
        
        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]

        coef1 = 1 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)
        mu = coef1 * (x - coef2 * e_pred)

        if t > 0:
            noise_step = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mu + sigma_t * noise_step
        else:
            x = mu

    x_denoised = x.clamp(0, 1).cpu()

    # Mostrar imágenes: original, con ruido, denoised
    fig, axes = plt.subplots(1, 3, figsize=(6, 2))
    axes[0].imshow(x_original[0, 0], cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(x_noisy[0, 0].cpu(), cmap='gray')
    axes[1].set_title('Noisy')
    axes[1].axis('off')

    axes[2].imshow(x_denoised[0, 0], cmap='gray')
    axes[2].set_title('Denoised')
    axes[2].axis('off')

    plt.show()

    return x_denoised



In [ ]:
def train(
    model, 
    stft_transform,
    epochs=1000, 
    batch_size=16, 
    lr=1e-3, 
    model_path=None,
    sch_path=None, 
    dataset=NSynth('training'), 
    verb=False
):
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,  pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs, device=device).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    optimizer = torch.optim.Adam(diffuser.parameters(), lr=lr)
    
    mse_loss = nn.MSELoss()
    best_loss = 1
    losses = []
    
    for epoch in range(epochs):
        for wave, _, _, _ in train_loader:
            # wave = wave.to(device)
            # x = stft_transform(wave)
            
            wave = wave.to(device)
            x = stft_transform(wave)
            batch_size = x.size(0)
            t = torch.randint(0, epochs, (batch_size,), device=device)

            z, e = diffuser(x, t)
            e_pred = model(z, t)

            loss = mse_loss(e_pred, e)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        losses.append(loss.item())
            
        if epoch % 10 == 0:
            if verb:
                print(f"Epoch {epoch}, Loss: {loss.item()}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            if model_path is not None:
                torch.save(model.state_dict(), model_path)
                torch.save(scheduler.state_dict(), sch_path)
                if verb:
                    print(f"Model temporally saved at epoch {epoch} with loss {best_loss}")
            else:
                torch.save(model.state_dict(), temp_path)
                torch.save(scheduler.state_dict(), sch_temp_path)
                if verb:
                    print(f"Model saved at epoch {epoch} with loss {best_loss}")
    
    if model_path is not None:
        model.load_state_dict(torch.load(model_path))
        scheduler.load_state_dict(torch.load(sch_path))
    else:
        model.load_state_dict(torch.load(temp_path))
        scheduler.load_state_dict(torch.load(sch_temp_path))

        # eliminar temp?
        
    print("Training completed., best loss:", best_loss)
    
    return model, scheduler, losses
    

In [ ]:
# STFT transform
sample_rate = 16000
n_fft = 1500
hop_length = 250
win_length = n_fft
stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=None, onesided=False, center=False
).to(device)
istft_transform = T.InverseSpectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length, onesided=False
).to(device)

# TRAIN SETUP
batch_size = 128
learning_rate = 1e-4
epochs = 100
train_loader = DataLoader(NSynth('training'), batch_size=batch_size, shuffle=True, pin_memory=True)
valid_loader = DataLoader(NSynth('validation'), batch_size=batch_size, shuffle=True, pin_memory=True)

# MODEL SETUP
input_height = 1500
input_width = 251
input_size = (input_height, input_width)
emb_dim = 128
norm_groups = 8


# layers
sin = 128
sout = 128

down_layers = [
    DummyLayer(sin, sin//2, norm_groups, emb_dim, skip=True).to(device),
    DummyLayer(sin//2, sin//4, norm_groups, emb_dim, stride=1).to(device),
]

bottleneck = DummyLayer(sin//4, sout//4, norm_groups, emb_dim).to(device)

# TODO podría hacer constructores de layers en el que les meto el numero de capas, el tamaño inicial y final y me devuelven la lista??
up_layers = [
    DummyLayer(sout//4, sout//2, norm_groups, emb_dim, stride=1).to(device),
    DummyLayer(sout//2 + sin//2, sout, norm_groups, emb_dim).to(device), # TODO hacer que el skip lo controle esta y la otra no
]

# embeder
embedder = Embeder(num_epochs=epochs, embed_dim=emb_dim, device=device).to(device)

# scheduler
scheduler = Scheduler(num_epochs=epochs, device=device).to(device)

# model
model = DiffusionModel(
    layer_channels=(sin, sout), # entrada y salida de las layers
    norm_groups=norm_groups,
    up_layers=up_layers,
    down_layers=down_layers,
    bottleneck=bottleneck,
    embedder=embedder,
    input_channels=1,
    output_channels=1        
).to(device)


In [ ]:
# %%time
model, scheduler, losses = train(
    model, 
    epochs=epochs,
    batch_size=batch_size, 
    lr=1e-3, 
    model_path=model_path, 
    sch_path=sch_path,
    dataset=NSynth('training'),
    verb=True,
    stft_transform=stft_transform
)
plt.plot(losses)

NameError: name 'mnist_ds' is not defined